In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

## Tensors

Tensors are very much like numpy.array's, just with some additional functionality (discussed later).<br>
They are used in a very similar way.

In [ ]:
x = torch.tensor([[1,2,3,4],[5,6,7,8]])
print(x)

In [ ]:
x += 2
print(x)

In [ ]:
print(x**2)

In [ ]:
y = torch.log(x)
print(y)

In [ ]:
x = torch.zeros(4,3)
print(x)

In [ ]:
x = torch.rand(3,5)
print(x)

One of the main features of tensors, is that we can use them to compute derivatives automatically.<br>
In the following example we will use tensors to compute the derivative of the function:
$$f(x,y) = A e^{-x}\sin(y) + B.$$
In particular, we will be looking for:
$$ \frac{\partial f}{\partial x}(4.1,-3.7),\quad\text{and}\quad, \frac{\partial f}{\partial y}(4.1,-3.7)$$
Note that in this example we have
$$ \frac{\partial f}{\partial x}(x,y)=-Ae^{-x}\sin(y),\quad\text{and}\quad, \frac{\partial f}{\partial y}(x,y) = A e^{-x} \cos(y).$$

In [ ]:
### Constants
A = 1.3
B = 2.2

### Setting 'required_grad=True' tells PyTorch that we will want to compute derivatives 
### with respect to these variables.
x = torch.tensor(4.1, requires_grad=True)
y = torch.tensor(-3.7, requires_grad=True)

### Defining the function f(x,y)
f = A*torch.exp(-x)*torch.sin(y)+B

### Calling f.backwards() makes PyTorch compute the derivatives via back-propagation.
f.backward()

### Printing the gradients computed by PyTorch.
print("Gradients computed by PyTorch:")
print(x.grad.item())
print(y.grad.item())

### Computing the derivatives manually.
fx = (-A*torch.exp(-x)*torch.sin(y)).detach().numpy()
fy = (A*torch.exp(-x)*torch.cos(y)).detach().numpy()

print("\nGradients computed manually:")
print(fx)
print(fy)

## Example - Polynomial Regression

The following example was borrowed from [Learning PyTorch with Examples](https://pytorch.org/tutorials/beginner/pytorch_with_examples.html), where you can look for more info/examples.

We try to approximate the function $\sin(x)$ using a polynomial of the form $$P(x)  = a+bx+cx^2+dx^3.$$
Our goal is to find the best values of $a,b,c,d$, using a gradient-descent algorithm.

In [ ]:
SEED = 22176
learning_rate = 1e-6
max_iter = 2000
print_iter = 200
N = 2000

# Create tensors to hold input and outputs.
x = torch.linspace(-math.pi, math.pi, N)
y = torch.sin(x)

## Step 1: using tensors

We use tensors to compute the derivatives of the error with respect to $a,b,c,d$, and then perform gradient descent.

In [ ]:
# Create random tensors for weights. 
# For a third order polynomial, we need 4 weights: y = a + b x + c x^2 + d x^3
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these tensors during the backward pass.

torch.manual_seed(SEED)
a = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
c = torch.randn(1, requires_grad=True)
d = torch.randn(1, requires_grad=True)

for t in range(max_iter):
 
    # Forward pass: compute predicted y using operations on tensors.
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss using operations on tensors.
    # Now loss is a tensor of shape (1,)
    # loss.item() gets the scalar value held in the loss.
    loss = torch.sum((y_pred - y)**2)
    if t % print_iter == 0:
        print(t, loss.item())

    # Use autograd to compute the backward pass. This call will compute the
    # gradient of loss with respect to all tensors with requires_grad=True.
    # After this call a.grad, b.grad. c.grad and d.grad will be tensors holding
    # the gradient of the loss with respect to a, b, c, d respectively.
    loss.backward()

    # Manually update weights using gradient descent. Wrap in torch.no_grad()
    # because weights have requires_grad=True, but we don't need to track this
    # in autograd.
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad

        # Manually zero the gradients after updating weights
        a.grad = None
        b.grad = None
        c.grad = None
        d.grad = None

print(f'\nResult: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

In [ ]:
plt.plot(x,y)
yy_1 = a.item() + b.item()*x + c.item()*x**2 + d.item()*x**3
plt.plot(x, yy_1)
plt.legend(['$\sin(x)$', '$a+bx+cx^2+dx^3$']);

## Step 2: using PyTorch optimisers

We  run the same example again, but instead of performing the gradient descent manually, we will use **torch.optim.SGD** to perform gradient descent for us.

In [ ]:
torch.manual_seed(SEED)

a = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
c = torch.randn(1, requires_grad=True)
d = torch.randn(1, requires_grad=True)

# Generating an SGD optimiser, to find the best fit for a,b,c,d.
optimiser = torch.optim.SGD([a,b,c,d], lr=learning_rate)

for t in range(max_iter):
 
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    loss = torch.sum((y_pred - y)**2)
    if t % print_iter == 0:
        print(t, loss.item())

    # Before the backward pass, use the optimizer object to zero all of the
    # gradients for the variables it will update (which are the learnable
    # weights of the model). This is because by default, gradients are
    # accumulated in buffers( i.e, not overwritten) whenever .backward()
    # is called. Checkout docs of torch.autograd.backward for more details.
    optimiser.zero_grad()

    # Backward pass: compute gradient of the loss with respect to model parameters
    loss.backward()

    # Calling the step function on an Optimizer makes an update to its parameters
    optimiser.step()

print(f'\nResult: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

In [ ]:
plt.plot(x,y)
yy_2 = a.item() + b.item()*x + c.item()*x**2 + d.item()*x**3
plt.plot(x, yy_2)
plt.legend(['$\sin(x)$', '$a+bx+cx^2+dx^3$']);

## Step 3: using PyTorch neural networks (manually constructed)

In [ ]:
class Polynomial3(nn.Module):
    def __init__(self):
        super().__init__()

        # In the constructor we instantiate four parameters and assign them as member parameters.
        self.a = nn.Parameter(torch.randn(1))
        self.b = nn.Parameter(torch.randn(1))
        self.c = nn.Parameter(torch.randn(1))
        self.d = nn.Parameter(torch.randn(1))

    def forward(self, x):
        
        # In the forward function we take a tensor of input data and return a tensor of output data. 
        # We can use Modules defined in the constructor as well as arbitrary operators on tensors.
        return self.a + self.b * x + self.c * x ** 2 + self.d * x ** 3

torch.manual_seed(SEED)

# Construct our model by instantiating the class defined above
poly = Polynomial3()

# Construct our loss function and an optimiser. The call to model.parameters()
# in the SGD constructor will contain the learnable parameters (defined 
# with torch.nn.Parameter) which are members of the model.
loss_fn = nn.MSELoss(reduction='sum')
optimiser = torch.optim.SGD(poly.parameters(), lr=learning_rate)

for t in range(max_iter):
    # Forward pass: Compute predicted y by passing x to the model
    y_pred = poly(x)

    # Compute and print loss
    loss = loss_fn(y_pred, y)
    
    if t % print_iter == 0:
        print(t, loss.item())

    # Zero gradients, perform a backward pass, and update the weights.
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

a = poly.a
b = poly.b
c = poly.c
d = poly.d
print(f'\nResult: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

In [ ]:
plt.plot(x,y)
# Note that in order to plot the values from a tensor, we first need to convert it to numpy array.
yy_3 = poly(x).detach().numpy()
plt.plot(x, yy_3)
plt.legend(['$\sin(x)$', '$a+bx+cx^2+dx^3$']);

## Step 4: using PyTorch neural network (automatically constructed)

In [ ]:
# Prepare the input tensor (x, x^2, x^3).
p = torch.tensor([1, 2, 3])
xx = x.unsqueeze(-1).pow(p)

torch.manual_seed(SEED)

# Use the nn package to define our model and loss function.
poly2 = nn.Sequential(
    nn.Linear(3, 1),
    nn.Flatten(0, 1)
)

# Loss fuction and optmiser as before.
loss_fn = nn.MSELoss(reduction='sum')
optimiser = torch.optim.SGD(poly2.parameters(), lr=learning_rate)

for t in range(max_iter):
    # Forward pass: compute predicted y by passing x to the model.
    y_pred = poly2(xx)

    # Compute and print loss.
    loss = loss_fn(y_pred, y)
    
    if t % print_iter == 0:
        print(t, loss.item())

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()


# The values of a,b,c,d can be read from the weights of the Linear layer in our network.
# Note that 'a' represents the bias term.

linear_layer = poly2[0]
a = linear_layer.bias
b = linear_layer.weight[:,0]
c = linear_layer.weight[:,1]
d = linear_layer.weight[:,2]

print(f'\nResult: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

In [ ]:
plt.plot(x,y)
yy_4 = poly2(xx).detach().numpy()
plt.plot(x, yy_4)
plt.legend(['$\sin(x)$', '$a+bx+cx^2+dx^3$']);

In [ ]:
plt.figure(figsize=(15,10))
plt.plot(x,y)
plt.plot(x, yy_1)
plt.plot(x, yy_2)
plt.plot(x, yy_3)
plt.plot(x, yy_4)
plt.legend(['$\sin(x)$', 'manual','optimiser', 'neural network (auto)', 'neural network (manual)']);

## Using a feedforward neural network

In [ ]:
# Defining nnsin as a feedforward NN.
# We take 4 linear layers with ReLU activation between them.
nnsin = nn.Sequential(
    nn.Linear(1, 10),
    nn.ReLU(),
    nn.Linear(10,10),
    nn.ReLU(),
    nn.Linear(10,5),
    nn.ReLU(),
    nn.Linear(5,1)
)

max_iter = 10000
print_iter = 1000
# Loss fuction and optmiser as before.
loss_fn = nn.MSELoss(reduction='sum')
optimiser = torch.optim.SGD(nnsin.parameters(), lr=1e-4)

for t in range(max_iter):
    # Forward pass: compute predicted y by passing x to the model.
    y_pred = nnsin(x[:, None])
  
    loss = loss_fn(y_pred, y[:,None])
    
    if t % print_iter == 0:
        print(t, loss.item())

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()


In [ ]:
plt.plot(x,y)
yy_5 = nnsin(x[:,None]).detach().numpy()
plt.plot(x, yy_5)